# Statistical significance checks on the pilot runs
50/100 questions, 20/50 template variations -> 1000/5000 total questions (20% of the benchmark)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
import pandas as pd
from statsmodels.formula.api import gee
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families.family import Binomial
from statsmodels.stats.multitest import multipletests
import numpy as np

from rpy2.rinterface_lib.embedded import RRuntimeError
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Set pandas converter as the global default for rpy2
ro.conversion.set_conversion(pandas2ri.converter + ro.default_converter)

from pymer4.models import glmer

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'mini_sep_new__20x50__20_12/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [ ]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
# mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

In [ ]:
# pea_sep = PromptEffectAnalyser(mres_standard, mres_sep, "Anti-babbling prompt")
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt")

In [ ]:
models = pea_code._baseline_mres.variants['main'].models
model = models[0]
model

In [ ]:
def prep_df(variant, model_):
    res = pea_code._baseline_mres.variants[variant].full_data
    res = res[res.model == model_][:]
    res = res[['id', 'correct']]
    res['is_variant'] = [int(variant!='GSM8K')] * len(res)
    res['is_correct'] = res['correct'].astype(int)
    res.drop('correct', inplace=True, axis=1)
    return res

df = pd.concat([prep_df('GSM8K', model), prep_df('main', model)]).reset_index(drop=True)
df


In [ ]:
# Your data structure:
# - question_id (1-100)
# - condition ('original' or 'variant')
# - correct (0 or 1)

model = gee("is_correct ~ is_variant", groups="id",
            data=df, family=Binomial(),
            cov_struct=Exchangeable())
result = model.fit()
print(result.summary())

In [ ]:
all_results = {}
for model in models:
    df = pd.concat([prep_df('GSM8K', model), prep_df('main', model)]).reset_index()

    glmm_model = gee("is_correct ~ is_variant", groups="id",
                     data=df, family=Binomial(),
                     cov_struct=Exchangeable())
    result = glmm_model.fit()
    all_results[model] = {'p_value': result.pvalues['is_variant']}

pd.DataFrame(all_results).T

In [ ]:
df_original = pea_code._baseline_mres.variants['GSM8K'].full_data
df_variants = pea_code._baseline_mres.variants['main'].full_data
df = pd.concat([df_original, df_variants]).reset_index(drop=True)
df

In [ ]:
difficulty = pea_code._baseline_mres.get_question_difficulty()

# Merge back into full dataframe
df = df.merge(difficulty.reset_index(), on='id', how='left')

# Add is_variant column
df['is_variant'] = (df['instance'] != -1).astype(int)

# correct as int (needed for modelling)
df['correct'] = df['correct'].astype(int)

df

In [ ]:
df_for_glmm = pd.DataFrame({
    'id': df['id'].astype(int),
    'correct': df['correct'].astype(int),
    'is_variant': df['is_variant'].astype(int),
    'difficulty': df['difficulty'].astype(float),
})

In [ ]:
df

In [ ]:
glmm_model = glmer(
    'correct ~ is_variant + difficulty + (1 | id)',
    data=df_for_glmm,
    family='binomial'
)
try:
    glmm_model.fit(verbose=False)  # fitting works, only getting stats fails
except RRuntimeError:
    pass

assert glmm_model.r_model is not None, "Model fitting failed entirely"

# Assign the model to an R variable first
ro.globalenv['r_model'] = glmm_model.r_model

# Then extract coefficients as a DataFrame
with localconverter(ro.default_converter + pandas2ri.converter):
    coefs_df = ro.r('as.data.frame(coef(summary(r_model)))')

coefs_df

In [ ]:
print(coefs_df)

## TO DO:
- workflow: use GLMM rather than GEE (below), but otherwise - same information needed (probably - review)
- use leave-one-out for difficulty - see Clause (Pymer4 GLMM chat, towards the end, but before last question)
- both accuracies
- plot matrix, not bars, of question difficulty
- clean up, move to pea
- add multiple comparisons correction (or already added)?
- analyse global effect
- new plots
- check on the full data

In [ ]:
%%sql


In [ ]:
glmm_results = []

for model_name, group_df in df.groupby('model'):
    group_df_for_glmm = pd.DataFrame({
        'id': group_df['id'].astype(int),
        'correct': group_df['correct'].astype(int),
        'is_variant': group_df['is_variant'].astype(int),
        'difficulty': group_df['difficulty'].astype(float),
    })

    glmm_model = glmer(
        'correct ~ is_variant + difficulty + (1 | id)',
        data=group_df_for_glmm,
        family='binomial'
    )
    try:
        glmm_model.fit(verbose=False)  # fitting works, only getting stats fails
    except RRuntimeError:
        pass

    if glmm_model.r_model is None:
        print(f"GLMM model fitting for LLM '{model_name}' failed entirely")
        continue

    # Assign the model to an R variable first
    ro.globalenv['r_model'] = glmm_model.r_model

    # Then extract coefficients as a DataFrame
    with localconverter(ro.default_converter + pandas2ri.converter):
        coefs_df = ro.r('as.data.frame(coef(summary(r_model)))')

    est = coefs_df.loc['is_variant', 'Estimate']
    pval = coefs_df.loc['is_variant', 'Pr(>|z|)']
    std_err = coefs_df.loc['is_variant', 'Std. Error']

    is_a_drop = est < 0  # Negative estimate means lower log-odds (lower accuracy)

    glmm_results.append({
        'model': model_name,
        'estimate': est,
        'odds_ratio': np.exp(est),
        'std_err': std_err,
        'p_value': pval,
        'drop': est < 0,
        'significant': pval < 0.05,
        'potentially_significant': pval < 0.15
    })

glmm_results_df = pd.DataFrame(glmm_results)

model_accuracy_drops = pea_code._baseline_mres.get_accuracy_drop_df('main').groupby('model').mean().reset_index()
glmm_results_df['accuracy_drop'] = model_accuracy_drops.accuracy_drop

# Calculate the Odds Ratio if you don't already have it
glmm_results_df['odds_ratio'] = np.exp(glmm_results_df['estimate'])

# --- 2. Calculate the 95% Confidence Intervals ---
# Calculate in log-odds space
glmm_results_df['ci_lower_log'] = glmm_results_df['estimate'] - 1.96 * glmm_results_df['std_err']
glmm_results_df['ci_upper_log'] = glmm_results_df['estimate'] + 1.96 * glmm_results_df['std_err']

# Exponentiate to get Odds Ratio CIs
glmm_results_df['ci_lower_or'] = np.exp(glmm_results_df['ci_lower_log'])
glmm_results_df['ci_upper_or'] = np.exp(glmm_results_df['ci_upper_log'])


# apply Benjamini-Hochberg procedure - controls the false discovery rate (multiple comparisons correction)
rejected, p_corrected, _, _ = multipletests(glmm_results_df['p_value'], method='fdr_bh')
glmm_results_df['p_corrected_bh'] = p_corrected
glmm_results_df['significant_bh'] = rejected

glmm_results_df

In [ ]:
import seaborn as sns
from matplotlib.lines import Line2D

# --- 4. Plotting the Forest Plot ---
fig, ax = plt.subplots(figsize=(8, 6))

# Sort the dataframe so the best performing models (highest OR) are at the top
df_plot = glmm_results_df.sort_values('odds_ratio', ascending=True)

def get_color(p):
    # Using a distinct colour palette: Dark Red for strong significance,
    # Orange for moderate, and Grey for non-significant
    if p < 0.01: return '#b2182b'      # Dark Red
    elif p < 0.05: return '#ef8a62'    # Orange/Coral
    else: return '#999999'             # Grey

df_plot['color'] = df_plot['p_value'].apply(get_color)


# Loop to plot CIs and Colored Dots
for i, row in df_plot.iterrows():
    # Draw CIs (we can color these grey, or match the dot color. Grey is usually cleaner)
    ax.hlines(y=row['model'], xmin=row['ci_lower_or'], xmax=row['ci_upper_or'],
              color='darkgrey', linewidth=2, zorder=1)

    # Draw the dot using the dynamically assigned color
    ax.scatter(x=row['odds_ratio'], y=row['model'],
               color=row['color'], s=100, zorder=2, edgecolor='black', linewidth=0.5)

# Line of no effect
ax.axvline(x=1, color='black', linestyle='--', linewidth=1.2, zorder=0)

# Format X-axis
ax.set_xscale('log')
ax.set_xlabel('Odds Ratio (Log Scale)', fontsize=11)
ax.set_ylabel('LLM Model', fontsize=11)
ax.set_title('Effect of Variant Condition on LLM Performance', fontsize=14, pad=15)

# --- 4. Custom Legend ---
# Because we colored dots dynamically, we need to manually build a legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#b2182b', markersize=10, label='p < 0.01 (Strong Drop)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ef8a62', markersize=10, label='p < 0.05 (Significant Drop)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#999999', markersize=10, label='p \u2265 0.05 (Not Significant)')
]
ax.legend(handles=legend_elements, loc='lower right', title="Significance", frameon=True)

sns.despine(left=True, bottom=True)
plt.tight_layout()

In [ ]:
ig, ax = plt.subplots()
ax.hist(difficulty, 21)
ax.set_xlabel("Question difficulty")
ax.set_ylabel("Question count")